In [28]:
import pandas as pd
import numpy as np
import openpyxl
import networkx
import matplotlib

In [14]:
# Change the path if needed
file_path = "Data_Analytics_Fresher_Practical_Test_2_Ecommerce.xlsx"

orders    = pd.read_excel(file_path, sheet_name="Ecom_Orders")
products  = pd.read_excel(file_path, sheet_name="Product_Master")
customers = pd.read_excel(file_path, sheet_name="Customer_Master")
refunds   = pd.read_excel(file_path, sheet_name="Refunds")
campaign  = pd.read_excel(file_path, sheet_name="Campaign_Performance")
traffic   = pd.read_excel(file_path, sheet_name="Web_Traffic")

print("Orders:", orders.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)
print("Refunds:", refunds.shape)
print("Campaign:", campaign.shape)
print("Traffic:", traffic.shape)

Orders: (2020, 13)
Products: (60, 8)
Customers: (420, 7)
Refunds: (170, 6)
Campaign: (120, 7)
Traffic: (120, 6)


#### C1: Clean & Merge

In [15]:
# Remove exact duplicates
orders  = orders.drop_duplicates(subset=["Order_ID"], keep="first")
refunds = refunds.drop_duplicates(subset=["Refund_ID"], keep="first")

In [21]:
# 2. Standardise text fields
def clean_status(x):
    if pd.isna(x):
        return "Unknown"
    x = str(x).strip().lower()
    if x in ["delivered"]:
        return "Delivered"
    if x in ["processing"]:
        return "Processing"
    if x in ["cancelled", "canceled"]:
        return "Cancelled"
    return x.title()

def clean_channel(x):
    if pd.isna(x) or str(x).strip() == "":
        return "Unknown"
    x = str(x).strip().lower()
    mapping = {
        "social": "Social", "social media": "Social",
        "organic": "Organic Search", "organic search": "Organic Search", "seo": "Organic Search",
        "website direct": "Direct", "direct": "Direct",
        "paid search": "Paid Search", "ppc": "Paid Search", "google ads": "Paid Search",
        "email": "Email", "newsletter": "Newsletter"
    }
    return mapping.get(x, x.title())

def clean_device(x):
    if pd.isna(x) or str(x).strip() == "":
        return "Unknown"
    x = str(x).strip().lower()
    if x in ["desktop", "pc"]:
        return "Desktop"
    if x in ["mobile"]:
        return "Mobile"
    if x in ["tablet"]:
        return "Tablet"
    return x.title()

# ---- Orders ----
orders["Order_Status"] = orders["Order_Status_Raw"].apply(clean_status)
orders["Channel"]      = orders["Marketing_Channel_Raw"].apply(clean_channel)
orders["Device"]       = orders["Device_Raw"].apply(clean_device)

# ---- Campaign (safe) ----
# Try the most likely column names
if "Marketing_Channel" in campaign.columns:
    campaign["Channel"] = campaign["Marketing_Channel"].apply(clean_channel)
elif "Channel_Raw" in campaign.columns:
    campaign["Channel"] = campaign["Channel_Raw"].apply(clean_channel)
else:
    # fallback – print available columns so you can see the real name
    print("Available Campaign columns:", campaign.columns.tolist())
    raise KeyError("Could not find channel column in Campaign sheet")

# ---- Traffic (safe) ----
if "Channel_Raw" in traffic.columns:
    traffic["Channel"] = traffic["Channel_Raw"].apply(clean_channel)
elif "Marketing_Channel" in traffic.columns:
    traffic["Channel"] = traffic["Marketing_Channel"].apply(clean_channel)
else:
    print("Available Traffic columns:", traffic.columns.tolist())
    raise KeyError("Could not find channel column in Traffic sheet")

print("Cleaning completed successfully!")
print("Orders Channel sample:", orders["Channel"].value_counts().head())
print("Campaign Channel sample:", campaign["Channel"].value_counts().head())

Cleaning completed successfully!
Orders Channel sample: Channel
Organic Search    501
Paid Search       417
Direct            397
Social            364
Email             200
Name: count, dtype: int64
Campaign Channel sample: Channel
Organic Search    24
Paid Search       24
Social            24
Direct            24
Email             15
Name: count, dtype: int64


In [ ]:

# Exclude invalid revenue rows (Cancelled or Qty <= 0)

orders["Valid"] = (orders["Order_Status"] != "Cancelled") & (orders["Quantity"] > 0)
valid_orders = orders[orders["Valid"]].copy()

# Merge required data
df = valid_orders.merge(
    products[["Product_ID", "Product_Name", "Category", "Brand", "List_Price", "Unit_Cost"]],
    on="Product_ID", how="left"
)

df = df.merge(
    customers[["Customer_ID", "Customer_Name", "Region", "Customer_Tier", "Signup_Date"]],
    on="Customer_ID", how="left"
)

# Flag refunded orders
refunded_ids = set(
    refunds.loc[refunds["Refund_Status"].isin(["Refunded", "Approved"]), "Order_ID"]
)
df["Is_Refunded"] = df["Order_ID"].isin(refunded_ids)

print("Cleaned & merged shape:", df.shape)
print("Valid orders:", df["Order_ID"].nunique())
df.head(3)

Cleaned & merged shape: (1914, 27)
Valid orders: 1914


,Order_ID,Order_Date,Customer_ID,Product_ID,Marketing_Channel_Raw,Device_Raw,Quantity,Coupon_Discount_Pct,Shipping_Fee,Payment_Mode,...,Product_Name,Category,Brand,List_Price,Unit_Cost,Customer_Name,Region,Customer_Tier,Signup_Date,Is_Refunded
0,EO202500152,2025-05-08,EC0043,EP018,Social,desktop,3,0.10,0,Credit Card,...,Table Lamp,Home & Kitchen,BluePeak,2150,1385.64,Nisha Kapoor,East,Regular,2025-09-02,False
1,EO202501187,2025-10-16,EC0101,EP001,Organic Search,Mobile,1,0.05,0,UPI,...,Bluetooth Speaker,Electronics,UrbanLeaf,30900,21114.63,Meera Das,South,New,2022-01-11,False
2,EO202401281,2024-02-03,EC0354,EP001,PPC,MOBILE,1,0.10,0,Debit Card,...,Bluetooth Speaker,Electronics,UrbanLeaf,30900,21114.63,Kabir Roy,South,Regular,2022-10-09,False


#### C2 & C3: Calculate Key Metrics (needed for C2 & C3)

In [23]:
# Product Revenue = Qty × List_Price × (1 - Discount)
df["Product_Revenue"] = df["Quantity"] * df["List_Price"] * (1 - df["Coupon_Discount_Pct"])

# Shipping Revenue
df["Shipping_Revenue"] = df["Shipping_Fee"].fillna(0)

# Net Revenue
df["Net_Revenue"] = df["Product_Revenue"] + df["Shipping_Revenue"]

# COGS
df["COGS"] = df["Quantity"] * df["Unit_Cost"]

# Gross Profit
df["Gross_Profit"] = df["Net_Revenue"] - df["COGS"]

# Net Unit Price (useful for refund value if needed later)
df["Net_Unit_Price"] = np.where(df["Quantity"] > 0, df["Product_Revenue"] / df["Quantity"], 0)

print("Metrics calculated successfully")
df[["Order_ID", "Net_Revenue", "Gross_Profit", "Is_Refunded"]].head()

Metrics calculated successfully


,Order_ID,Net_Revenue,Gross_Profit,Is_Refunded
0,EO202500152,5805.0,1648.08,False
1,EO202501187,29355.0,8240.37,False
2,EO202401281,27810.0,6695.37,False
3,EO202500541,5355.0,1714.98,False
4,EO202501722,4465.0,1361.19,False


#### C2: Channel Performance + Export CSV

In [32]:
from networkx import display
# Channel-level aggregation
channel_perf = df.groupby("Channel").agg(
    Orders          = ("Order_ID", "nunique"),
    Net_Revenue     = ("Net_Revenue", "sum"),
    Gross_Profit    = ("Gross_Profit", "sum"),
    Refunded_Orders = ("Is_Refunded", "sum")
).reset_index()

# Refund Rate %
channel_perf["Refund_Rate_%"] = (
    channel_perf["Refunded_Orders"] / channel_perf["Orders"] * 100
).round(2)

# Marketing Spend
spend = campaign.groupby("Channel")["Marketing_Spend"].sum().reset_index()
channel_perf = channel_perf.merge(spend, on="Channel", how="left")
channel_perf["Marketing_Spend"] = channel_perf["Marketing_Spend"].fillna(0)

# ROAS = Net Revenue / Marketing Spend
channel_perf["ROAS"] = np.where(
    channel_perf["Marketing_Spend"] > 0,
    (channel_perf["Net_Revenue"] / channel_perf["Marketing_Spend"]).round(3),
    np.nan
)

# Sessions
sessions = traffic.groupby("Channel")["Sessions"].sum().reset_index()
channel_perf = channel_perf.merge(sessions, on="Channel", how="left")
channel_perf["Sessions"] = channel_perf["Sessions"].fillna(0)

# Order Conversion Rate %
channel_perf["Order_Conversion_Rate_%"] = np.where(
    channel_perf["Sessions"] > 0,
    (channel_perf["Orders"] / channel_perf["Sessions"] * 100).round(4),
    0
)

# Round financial columns
channel_perf["Net_Revenue"]  = channel_perf["Net_Revenue"].round(0)
channel_perf["Gross_Profit"] = channel_perf["Gross_Profit"].round(0)
channel_perf["Marketing_Spend"] = channel_perf["Marketing_Spend"].round(0)

# Sort by Net Revenue descending
channel_perf = channel_perf.sort_values("Net_Revenue", ascending=False).reset_index(drop=True)

# Final column order
channel_perf = channel_perf[[
    "Channel", "Orders", "Net_Revenue", "Gross_Profit",
    "Refunded_Orders", "Refund_Rate_%",
    "Marketing_Spend", "ROAS",
    "Sessions", "Order_Conversion_Rate_%"
]]

print("=== Channel Performance ===")
print("=== Channel Performance ===")
print(channel_perf)          # or simply leave channel_perf as the last line

# Export
channel_perf.to_csv("Channel_Performance.csv", index=False)
print("\n✅ Exported → Channel_Performance.csv")

=== Channel Performance ===
=== Channel Performance ===
          Channel  Orders  Net_Revenue  Gross_Profit  Refunded_Orders  \
0  Organic Search     482    6126170.0     1904574.0               17   
1     Paid Search     403    5235148.0     1555491.0               29   
2          Social     346    4851251.0     1565915.0               21   
3          Direct     378    4600567.0     1443859.0               18   
4           Email     192    2462360.0      726977.0               15   
5      Newsletter      80    1157673.0      355283.0                4   
6         Unknown      33     454687.0      139876.0                3   

   Refund_Rate_%  Marketing_Spend   ROAS  Sessions  Order_Conversion_Rate_%  
0           3.53        6087400.0  1.006  669966.0                   0.0719  
1           7.20       15908900.0  0.329  489222.0                   0.0824  
2           6.07        9776300.0  0.496  405699.0                   0.0853  
3           4.76        1284200.0  3.582  52971

#### C3: Short Analysis

In [35]:
# Top 5 Products by Gross Profit
top5_products = (
    df.groupby(["Product_ID", "Product_Name", "Category"])
      .agg(Gross_Profit=("Gross_Profit", "sum"),
           Net_Revenue=("Net_Revenue", "sum"),
           Orders=("Order_ID", "nunique"))
      .reset_index()
      .sort_values("Gross_Profit", ascending=False)
      .head(5)
)

top5_products["Gross_Profit"] = top5_products["Gross_Profit"].round(0)
top5_products["Net_Revenue"]  = top5_products["Net_Revenue"].round(0)

print("=" * 60)
print("(a) Top 5 Products by Gross Profit")
print("=" * 60)
print(top5_products)



# Customers with ≥ 4 valid orders AND Refund Rate > 20%
cust = df.groupby("Customer_ID").agg(
    Customer_Name   = ("Customer_Name", "first"),
    Region          = ("Region", "first"),
    Order_Count     = ("Order_ID", "nunique"),
    Refunded_Orders = ("Is_Refunded", "sum"),
    Net_Revenue     = ("Net_Revenue", "sum")
).reset_index()

cust["Refund_Rate_%"] = (cust["Refunded_Orders"] / cust["Order_Count"] * 100).round(2)

high_risk_customers = cust[
    (cust["Order_Count"] >= 4) &
    (cust["Refund_Rate_%"] > 20)
].sort_values("Refund_Rate_%", ascending=False)

high_risk_customers["Net_Revenue"] = high_risk_customers["Net_Revenue"].round(0)

print("\n" + "=" * 60)
print("(b) Customers with ≥ 4 valid orders AND Refund Rate > 20%")
print("=" * 60)
print(high_risk_customers)

print(f"\nTotal such customers: {len(high_risk_customers)}")

(a) Top 5 Products by Gross Profit
   Product_ID         Product_Name        Category  Gross_Profit  Net_Revenue  \
6       EP007        Action Camera     Electronics      610810.0    1712475.0   
7       EP008         Gaming Mouse     Electronics      598319.0    1393605.0   
8       EP009  Mechanical Keyboard     Electronics      532170.0    1783408.0   
16      EP017      Electric Kettle  Home & Kitchen      485168.0    1196445.0   
3       EP004          Tablet Mini     Electronics      468711.0    1534282.0   

    Orders  
6       29  
7       38  
8       32  
16      30  
3       44  

(b) Customers with ≥ 4 valid orders AND Refund Rate > 20%
    Customer_ID  Customer_Name Region  Order_Count  Refunded_Orders  \
248      EC0251  Ananya Kapoor   East            6                4   
58       EC0059    Nisha Verma  North            5                2   
335      EC0338   Aarav Sharma   West            6                2   
140      EC0143     Kabir Iyer   West            6       